# Step 16 — cohort B: projection and federation

Reads `step12_panels.rds`. Writes `step16_projection.rds`.

Cohort B has one surviving module, so its column axis is a single block. Two ways to give B the
structure the other cohorts found, and they are **not** the same thing:

- **Projection** carries a module *definition* into B. Loadings come from the source cohort and are
  never recomputed; the new samples are scaled with the **source's** per-protein mean and sd, not
  their own. Re-centring on B would move B's mean to zero, which is precisely the difference being
  measured. The round-trip gate must return r = 1.0000 or the code is refitting, not projecting.
- **Federation** pools sufficient statistics so no site sees another's patients. It is governed by
  the release rule `p < n`, because `rank(G) = min(n, p)` and a Gram matrix from a panel wider than
  the cohort can be inverted back toward individual rows.

In [ ]:
source("../src/paths.R")
suppressMessages(library(WGCNA))
options(stringsAsFactors = FALSE); set.seed(42)
P <- readRDS(art("step12_panels.rds"))
D <- P$D; spec <- P$spec; SITES <- P$SITES
W  <- lapply(SITES, function(s) readRDS(art("wgcna_%s.rds", s))); names(W) <- SITES

build_traits <- function(m, spec) {
  out <- data.frame(row.names = rownames(m))
  for (i in seq_len(nrow(spec))) {
    v <- m[[spec$source_column[i]]]
    out[[spec$name[i]]] <- switch(spec$type[i],
      numeric = as.numeric(as.character(v)),
      binary  = as.numeric(v == "Positive"),
      ordinal = { lvl <- unique(v[!is.na(v) & v != ""])
                  lvl <- lvl[order(as.numeric(sub("-.*", "", lvl)))]
                  as.integer(factor(v, levels = lvl, ordered = TRUE)) })
  }
  out
}
TR <- lapply(SITES, function(s) build_traits(W[[s]]$meta, spec)); names(TR) <- SITES
N_MIN <- min(sapply(W, function(w) nrow(w$X)))

suppressMessages(library(WGCNA)); options(stringsAsFactors=FALSE); set.seed(42)

# ---- projection: loadings from the SOURCE only, never recomputed ---------
project_module <- function(md0, X, mods, NEW){
  g  <- colnames(X)[mods==md0]
  mu <- colMeans(X[,g,drop=FALSE]); sdv <- apply(X[,g,drop=FALSE],2,sd)
  keep <- sdv>0; g<-g[keep]; mu<-mu[keep]; sdv<-sdv[keep]
  Zs <- scale(X[,g,drop=FALSE], center=mu, scale=sdv)
  v  <- svd(Zs, nu=0, nv=1)$v[,1]
  e  <- as.vector(Zs %*% v)
  if (cor(e, rowMeans(Zs)) < 0){ v <- -v; e <- -e }
  Zn <- scale(as.matrix(NEW[,g,drop=FALSE]), center=mu, scale=sdv)
  list(sle=e, new=as.vector(Zn %*% v))
}

cat("=== ROUTE 1: PROJECTION -- B scored on A's and C's module definitions ===\n")
proj <- list()
for (src in c("A","C")){
  d <- D[[paste(src,"all15")]]; Xs <- W[[src]]$X; mods <- W[[src]]$mods
  Xb <- W$B$X
  ME <- moduleEigengenes(Xs, mods)$eigengenes
  Pm <- lapply(d$keep, function(k) project_module(k, Xs, mods, Xb)); names(Pm) <- d$keep
  gate <- sapply(d$keep, function(k) abs(cor(Pm[[k]]$sle, ME[[paste0("ME",k)]])))
  cat(sprintf("\n  %s -> B : %d modules, round-trip r min %.4f median %.4f\n",
              src, length(d$keep), min(gate), median(gate)))
  stopifnot(min(gate) > 0.99)
  cat("  GATE PASSED -- projection, not a refit\n")
  trB <- TR$B[rownames(Xb), , drop=FALSE]
  sc  <- sapply(d$keep, function(k) Pm[[k]]$new); rownames(sc) <- rownames(Xb)
  r   <- cor(sc, trB, use="pairwise.complete.obs")
  p   <- 2*pt(-abs(r*sqrt((nrow(sc)-2)/(1-r^2))), nrow(sc)-2)
  q   <- matrix(p.adjust(p,"BH"), nrow=nrow(p), dimnames=dimnames(p))
  h   <- which(q<0.05, arr.ind=TRUE)
  cat(sprintf("  %s's modules tested against B's traits: %d of %d at FDR 5%%\n",
              src, nrow(h), length(q)))
  if (nrow(h)) print(data.frame(module=rownames(r)[h[,1]], trait=colnames(r)[h[,2]],
      r=round(r[h],2), q=signif(q[h],2), n_prot=sapply(rownames(r)[h[,1]],
      function(k) sum(mods==k)))[order(-abs(r[h])),], row.names=FALSE)
  proj[[src]] <- list(scores=sc, gate=gate, r=r, q=q)
}

cat("\n\n=== ROUTE 2: FEDERATION -- release rule p < n ===\n")
cat(sprintf("  smallest cohort n = %d, so a releasable panel needs p <= %d proteins\n\n", N_MIN, N_MIN-1))
fed <- do.call(rbind, lapply(names(D), function(nm){ d<-D[[nm]]
  data.frame(cohort=d$cohort, cond=d$cond, probes=length(d$sel),
             proteins=n_proteins(d$sel), n=N_MIN,
             releasable=n_proteins(d$sel) < N_MIN) }))
print(fed, row.names=FALSE)
cat(sprintf("\n  panels that pass: %d of %d. Federation of these panels is NOT possible.\n",
            sum(fed$releasable), nrow(fed)))
cat("  (step 10 federates an 80-protein panel exactly; none of these is that small.)\n")

cat("\n\n=== DIAGNOSTIC: why is A the most discriminating? ===\n")
cat("\nmodules and associations per cohort (all15):\n")
print(do.call(rbind, lapply(SITES, function(s){ d<-D[[paste(s,"all15")]]
  m <- W[[s]]$mods
  data.frame(cohort=s, n=nrow(W[[s]]$X), modules=length(setdiff(unique(m),"grey")),
    grey=sum(m=="grey"), largest=max(table(m[m!="grey"])),
    largest_pct=round(100*max(table(m[m!="grey"]))/7288,1),
    associated=length(d$sig)) })), row.names=FALSE)

cat("\ntrait spread per cohort (sd for numeric, positive count for binary):\n")
sp <- do.call(rbind, lapply(spec$name, function(t){
  row <- data.frame(trait=t, type=spec$type[spec$name==t])
  for (s in SITES){ v <- TR[[s]][[t]]
    row[[s]] <- if (spec$type[spec$name==t]=="binary") sum(v==1,na.rm=TRUE) else round(sd(v,na.rm=TRUE),2) }
  row }))
print(sp, row.names=FALSE)

cat("\nbatch composition (the split was stratified on Batch only):\n")
print(do.call(rbind, lapply(SITES, function(s)
  data.frame(cohort=s, t(as.matrix(table(W[[s]]$meta$Batch)))))), row.names=FALSE)

saveRDS(list(proj = proj, fed = fed), art("step16_projection.rds"))


## Findings

**Projection works, and it recovers the axis B found on its own.** The gate passes at r = 1.0000
both ways. A→B gives 5 associations, **4 of them on `bisque4`, a 10-protein module** — uPCR,
SLEDAI-2K, lymphocyte count, creatinine. C→B gives 2, both on `brown`, the same renal axis. B's own
single module is also renal. Three independent routes agree on what B contains.

**Federation of these panels is impossible.** 0 of 9 pass `p < n = 86`; the smallest is 430
proteins. Step 10 federates an 80-protein panel exactly, to 4.4e-12 — none of these is that small.
Getting there is a decision about what to give up, not a technicality.